# FedAvg — H&M Sampled Dataset

Federated Averaging (McMahan et al. 2017): each round, a random subset of
clients train locally for several epochs, then the server averages all model
parameters (user + item embeddings + biases) weighted by local dataset size.


In [2]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import math
import os
import itertools
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


## 1. Load Data


In [4]:
TARGET_USERS     = 1000
MIN_INTERACTIONS = 20
N_QUANTILES      = 4
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/7/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_INTERACTIONS}_t25_v20_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cache not found in '{SAMPLED_DATA_DIR}/'. Run sampling notebook first."
)

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded : {cache_tag}")
print(f"Users  : {n_users} | Items : {n_items}")
print(f"Train  : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")


Loaded : u1000_m20_t25_v20_s42
Users  : 1000 | Items : 10354
Train  : 63081 | Val : 15010 | Test : 25055


## 2. Build Evaluation Matrices & Per-User Split


In [6]:
min_rating = float(train_df["bought"].min())
max_rating = float(train_df["bought"].max())

def make_matrix(df, n_u, n_i, min_r, max_r):
    mat   = torch.full((n_u, n_i), -1.0)
    users = torch.tensor(df["customer_id"].values,  dtype=torch.long)
    items = torch.tensor(df["product_code"].values, dtype=torch.long)
    vals  = torch.tensor((df["bought"].values - min_r) / (max_r - min_r),
                         dtype=torch.float32)
    mat[users, items] = vals
    return mat

train_matrix = make_matrix(train_df, n_users, n_items, min_rating, max_rating)
val_matrix   = make_matrix(val_df,   n_users, n_items, min_rating, max_rating)
test_matrix  = make_matrix(test_df,  n_users, n_items, min_rating, max_rating)

print(f"Train : {(train_matrix != -1).sum().item()} observed")
print(f"Val   : {(val_matrix   != -1).sum().item()} observed")
print(f"Test  : {(test_matrix  != -1).sum().item()} observed")

# Per-user data split for local training
split_train = {}
for uid in range(n_users):
    split_train[uid] = train_df[train_df['customer_id'] == uid].reset_index(drop=True)
print(f"Users with training data: {sum(len(v)>0 for v in split_train.values())}")


Train : 63081 observed
Val   : 15010 observed
Test  : 25055 observed
Users with training data: 1000


## 3. Dataset, Model & Helpers


In [8]:
class HMDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        user = torch.tensor(int(row.customer_id),  dtype=torch.long)
        item = torch.tensor(int(row.product_code), dtype=torch.long)
        y    = torch.tensor(float(row.bought),     dtype=torch.float32)
        return user, item, y


class HMModel(nn.Module):
    """MF with user/item embeddings and biases."""
    def __init__(self, n_users, n_items, k=30):
        super().__init__()
        self.user_emb  = nn.Embedding(n_users, k)
        self.item_emb  = nn.Embedding(n_items, k)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        nn.init.normal_(self.user_emb.weight,  std=0.01)
        nn.init.normal_(self.item_emb.weight,  std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        u  = self.user_emb(user)
        v  = self.item_emb(item)
        ub = self.user_bias(user).squeeze(-1)
        vb = self.item_bias(item).squeeze(-1)
        return (u * v).sum(-1) + ub + vb


def compute_rmse(matrix, predicted, min_r, max_r):
    mask    = (matrix != -1).float()
    n       = mask.sum()
    if n == 0: return float('nan')
    pred_sc = predicted * (max_r - min_r) + min_r
    true_sc = matrix    * (max_r - min_r) + min_r
    return math.sqrt((((pred_sc - true_sc) ** 2 * mask).sum() / n).item())

val_dataset = HMDataset(val_df)
val_loader  = DataLoader(val_dataset, batch_size=256, shuffle=False)


def validate(models, loader):
    """Per-sample RMSE across val set."""
    for m in models.values(): m.eval()
    sq, n = 0.0, 0
    with torch.no_grad():
        for user, item, y in loader:
            uid   = int(user[0])   # batch_size=1 or 256; model keyed by uid
            # For batch evaluation use the global model (model_dict[0]) if uid unknown
            logit = models.get(uid, models[0])(user, item)
            sq   += ((logit - y) ** 2).sum().item()
            n    += len(y)
    return math.sqrt(sq / max(n, 1))


print("HMDataset, HMModel, validate defined.")


HMDataset, HMModel, validate defined.


## 4. Parameter Tuning -- Grid Search

Searches `lr`, `k` (latent dimension), and `local_epochs`.
Runs a short FedAvg simulation (few rounds, small client subset) to find
the combo with the lowest validation RMSE.
Run **once** after building matrices.


In [10]:
LR_GRID          = [0.001, 0.01, 0.05]
K_GRID           = [10,    20,   40  ]
LOCAL_EPOCH_GRID  = [1,     3,    5   ]
TUNE_ROUNDS       = 5
TUNE_N_CLIENTS    = min(20, n_users)

param_grid3 = list(itertools.product(LR_GRID, K_GRID, LOCAL_EPOCH_GRID))
print(f"Grid search: {len(param_grid3)} combos x {TUNE_ROUNDS} rounds")
print(f"{'#':>3}  {'lr':>7}  {'K':>4}  {'loc_ep':>7}  {'val RMSE':>10}")
print("-" * 38)
grid_results3 = []

for _k, (_lr, _K, _le) in enumerate(param_grid3, 1):
    torch.manual_seed(0)
    _models = {uid: HMModel(n_users, n_items, k=_K) for uid in range(n_users)}
    _opts   = {uid: torch.optim.SGD(_models[uid].parameters(), lr=_lr)
               for uid in range(n_users)}
    _crit   = nn.MSELoss()

    for _r in range(TUNE_ROUNDS):
        # Select random subset
        _selected = np.random.choice(n_users, TUNE_N_CLIENTS, replace=False)
        _weights  = np.array([max(len(split_train[u]), 1) for u in _selected], dtype=float)
        _weights /= _weights.sum()

        # Local training
        for _i, _uid in enumerate(_selected):
            if len(split_train[_uid]) == 0: continue
            _dl = DataLoader(HMDataset(split_train[_uid]), batch_size=32, shuffle=True)
            _models[_uid].train()
            for _ in range(_le):
                for _u, _v, _y in _dl:
                    _opts[_uid].zero_grad()
                    _loss = _crit(_models[_uid](_u, _v), _y)
                    _loss.backward()
                    _opts[_uid].step()

        # FedAvg aggregation
        _params = {}
        for _i, _uid in enumerate(_selected):
            for _name, _param in _models[_uid].named_parameters():
                if _name not in _params:
                    _params[_name] = _param.data.clone() * _weights[_i]
                else:
                    _params[_name] += _param.data * _weights[_i]
        for _uid in range(n_users):
            for _name, _param in _models[_uid].named_parameters():
                _param.data.copy_(_params[_name])

    _val = validate(_models, val_loader)
    grid_results3.append((_val, _lr, _K, _le))
    _marker = ' <--' if _val == min(r[0] for r in grid_results3) else ''
    print(f"{_k:>3}  {_lr:>7.4f}  {_K:>4d}  {_le:>7d}  {_val:>10.4f}{_marker}")

best_val_t, best_lr, best_K, best_le = min(grid_results3, key=lambda x: x[0])
print(f"\nBest val RMSE  : {best_val_t:.4f}")
print(f"  lr           = {best_lr}")
print(f"  k            = {best_K}")
print(f"  local_epochs = {best_le}")
lr           = best_lr
k            = best_K
local_epochs = best_le
print("\nHyperparameters updated. Run the training cell.")


Grid search: 27 combos x 5 rounds
  #       lr     K   loc_ep    val RMSE
--------------------------------------
  1   0.0010    10        1      1.7723 <--
  2   0.0010    10        3      1.7722 <--
  3   0.0010    10        5      1.7720 <--
  4   0.0010    20        1      1.7723
  5   0.0010    20        3      1.7721
  6   0.0010    20        5      1.7720 <--
  7   0.0010    40        1      1.7723
  8   0.0010    40        3      1.7721
  9   0.0010    40        5      1.7720
 10   0.0100    10        1      1.7718 <--
 11   0.0100    10        3      1.7709 <--
 12   0.0100    10        5      1.7693 <--
 13   0.0100    20        1      1.7717
 14   0.0100    20        3      1.7709
 15   0.0100    20        5      1.7703
 16   0.0100    40        1      1.7717
 17   0.0100    40        3      1.7705
 18   0.0100    40        5      1.7699
 19   0.0500    10        1      1.7701
 20   0.0500    10        3      1.7667 <--
 21   0.0500    10        5      1.7638 <--
 22   0.050

## 5. Training


In [12]:
try: lr
except NameError: lr = 0.01
try: k
except NameError: k = 30
try: local_epochs
except NameError: local_epochs = 5
num_rounds    = 50
patience      = 10
num_sampled   = min(200, n_users)   # clients selected per round

torch.manual_seed(0)
model_dict = {uid: HMModel(n_users, n_items, k=k) for uid in range(n_users)}
opt_dict   = {uid: torch.optim.SGD(model_dict[uid].parameters(), lr=lr)
              for uid in range(n_users)}
criterion  = nn.MSELoss()

best_val_rmse, patience_count, best_states = float('inf'), 0, None

for rnd in range(num_rounds):
    selected = np.random.choice(n_users, num_sampled, replace=False)
    weights  = np.array([max(len(split_train[u]), 1) for u in selected], dtype=float)
    weights /= weights.sum()

    # Local training
    for uid in selected:
        if len(split_train[uid]) == 0: continue
        dl = DataLoader(HMDataset(split_train[uid]), batch_size=32, shuffle=True)
        model_dict[uid].train()
        for _ in range(local_epochs):
            for u, v, y in dl:
                opt_dict[uid].zero_grad()
                loss = criterion(model_dict[uid](u, v), y)
                loss.backward()
                opt_dict[uid].step()

    # FedAvg aggregation
    params = {}
    for i, uid in enumerate(selected):
        for name, param in model_dict[uid].named_parameters():
            if name not in params:
                params[name] = param.data.clone() * weights[i]
            else:
                params[name] += param.data * weights[i]
    for uid in range(n_users):
        for name, param in model_dict[uid].named_parameters():
            param.data.copy_(params[name])

    val_rmse = validate(model_dict, val_loader)
    print(f"Round {rnd+1:3d} | val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_states   = {uid: {n: p.data.clone() for n, p in model_dict[uid].named_parameters()}
                         for uid in range(n_users)}
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"Early stopping at round {rnd+1}.")
            break

for uid in range(n_users):
    for name, param in model_dict[uid].named_parameters():
        param.data.copy_(best_states[uid][name])
print(f"\nBest val RMSE: {best_val_rmse:.4f}")


Round   1 | val RMSE: 1.7708
Round   2 | val RMSE: 1.7691
Round   3 | val RMSE: 1.7676
Round   4 | val RMSE: 1.7661
Round   5 | val RMSE: 1.7644
Round   6 | val RMSE: 1.7628
Round   7 | val RMSE: 1.7613
Round   8 | val RMSE: 1.7598
Round   9 | val RMSE: 1.7583
Round  10 | val RMSE: 1.7568
Round  11 | val RMSE: 1.7552
Round  12 | val RMSE: 1.7537
Round  13 | val RMSE: 1.7521
Round  14 | val RMSE: 1.7505
Round  15 | val RMSE: 1.7489
Round  16 | val RMSE: 1.7473
Round  17 | val RMSE: 1.7458
Round  18 | val RMSE: 1.7441
Round  19 | val RMSE: 1.7425
Round  20 | val RMSE: 1.7409
Round  21 | val RMSE: 1.7393
Round  22 | val RMSE: 1.7379
Round  23 | val RMSE: 1.7364
Round  24 | val RMSE: 1.7348
Round  25 | val RMSE: 1.7333
Round  26 | val RMSE: 1.7318
Round  27 | val RMSE: 1.7302
Round  28 | val RMSE: 1.7288
Round  29 | val RMSE: 1.7272
Round  30 | val RMSE: 1.7257
Round  31 | val RMSE: 1.7242
Round  32 | val RMSE: 1.7227
Round  33 | val RMSE: 1.7212
Round  34 | val RMSE: 1.7197
Round  35 | va

## 6. Test Evaluation


In [14]:
test_dataset = HMDataset(test_df)
test_loader  = DataLoader(test_dataset, batch_size=256, shuffle=False)

for m in model_dict.values(): m.eval()
sq, n = 0.0, 0
abs_err = 0.0
with torch.no_grad():
    for user, item, y in test_loader:
        uid   = int(user[0])
        logit = model_dict.get(uid, model_dict[0])(user, item)
        sq      += ((logit - y) ** 2).sum().item()
        abs_err += torch.abs(logit - y).sum().item()
        n       += len(y)
print(f"Test RMSE : {math.sqrt(sq/max(n,1)):.4f}")
print(f"Test MAE  : {abs_err/max(n,1):.4f}")


Test RMSE : 1.7124
Test MAE  : 1.3756
